# Demo 2: Error Scenarios (Non-Technical Walkthrough)\n
\n
## Objective\n
Show how the API protects data and business rules:\n
1. Rejects missing credentials (`401`)\n
2. Rejects orders without stock (`409`)\n
3. Rejects invalid payload (`422`)\n
\n
## Before you start\n
Run these commands in terminal: \n
- `make up`\n
- `make health`

In [4]:
BASE_URL = "http://localhost:8000"
BASE_URL

'http://localhost:8000'

In [ ]:
import json
import uuid

import requests

def show_response(label, response):
    print(f"\n=== {label} ===")
    print("Status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    except Exception:
        print(response.text)

## Setup: create user, login, and low-stock product

In [ ]:
run_id = uuid.uuid4().hex[:8]\n
email = f"demo_error_{run_id}@example.com"\n
password = "StrongPass123"\n
\n
payload = {"email": email, "password": password}\n
requests.post(f"{BASE_URL}/auth/register", json=payload, timeout=15)\n
r_login = requests.post(f"{BASE_URL}/auth/login", json=payload, timeout=15)\n
show_response("Login", r_login)\n
token = r_login.json().get("access_token")\n
headers = {"Authorization": f"Bearer {token}"}\n
\n
product_id = f"low_{run_id}"\n
product_payload = {\n
    "product_id": product_id,\n
    "product_name": "Low Stock Product",\n
    "quantity": 1\n
}\n
r_create = requests.post(f"{BASE_URL}/products", json=product_payload, timeout=15)\n
show_response("Create Low-Stock Product", r_create)

## 1) Missing Credentials Case\n
Expected: **401** with `no valid credentials`.

In [ ]:
r_no_auth = requests.post(\n
    f"{BASE_URL}/orders",\n
    json={"product_id": product_id, "quantity": 1},\n
    timeout=15,\n
)\n
show_response("Order Without Credentials", r_no_auth)

## 2) No Stock Case\n
We first consume the single available unit, then try one more order.\n
Expected second call: **409** with `no_stock`.

In [ ]:
r_first_order = requests.post(\n
    f"{BASE_URL}/orders",\n
    json={"product_id": product_id, "quantity": 1},\n
    headers=headers,\n
    timeout=15,\n
)\n
show_response("First Valid Order (consumes stock)", r_first_order)\n
\n
r_no_stock = requests.post(\n
    f"{BASE_URL}/orders",\n
    json={"product_id": product_id, "quantity": 1},\n
    headers=headers,\n
    timeout=15,\n
)\n
show_response("Second Order (no stock)", r_no_stock)

## 3) Invalid Payload Case\n
Expected: **422** for invalid quantity (negative number).

In [ ]:
r_invalid = requests.post(\n
    f"{BASE_URL}/orders",\n
    json={"product_id": product_id, "quantity": -3},\n
    headers=headers,\n
    timeout=15,\n
)\n
show_response("Invalid Payload", r_invalid)

## Non-Technical Interpretation\n
- Unauthorized users cannot place orders.\n
- The system blocks overselling.\n
- Bad input is rejected before affecting data.